# Demo 1: Exploring Temporal Patterns in Health Data

In this demo, we'll explore common patterns in health time series data using Python. We'll look at:
1. Loading and preprocessing time series data
2. Visualizing temporal patterns
3. Decomposing time series into components
4. Handling missing values and irregularities

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose

# Set style for better visualizations
plt.style.use('seaborn')
sns.set_palette('husl')

## 1. Generate Sample Health Data

Let's create some synthetic patient vital signs data to demonstrate time series concepts.

In [ ]:
# Create date range for one week of hourly measurements
dates = pd.date_range(start='2024-01-01', end='2024-01-07', freq='H')

# Generate synthetic vital signs with daily patterns
np.random.seed(42)

# Base pattern for heart rate (higher during day, lower at night)
hour_effect = 10 * np.sin(2 * np.pi * (dates.hour - 6) / 24)

# Create DataFrame with vital signs
vitals = pd.DataFrame({
    'timestamp': dates,
    'heart_rate': 70 + hour_effect + np.random.normal(0, 3, len(dates)),
    'temperature': 37 + 0.3 * np.sin(2 * np.pi * (dates.hour - 4) / 24) + np.random.normal(0, 0.1, len(dates)),
    'blood_pressure_systolic': 120 + 5 * np.sin(2 * np.pi * (dates.hour - 8) / 24) + np.random.normal(0, 2, len(dates))
})

# Set timestamp as index
vitals = vitals.set_index('timestamp')

# Display first few rows
vitals.head()

## 2. Visualize Daily Patterns

Let's examine how vital signs vary throughout the day.

In [ ]:
# Create figure with subplots
fig, axes = plt.subplots(3, 1, figsize=(12, 10))
fig.suptitle('Daily Patterns in Vital Signs')

# Plot heart rate
sns.boxplot(data=vitals, x=vitals.index.hour, y='heart_rate', ax=axes[0])
axes[0].set_title('Heart Rate by Hour')
axes[0].set_xlabel('Hour of Day')

# Plot temperature
sns.boxplot(data=vitals, x=vitals.index.hour, y='temperature', ax=axes[1])
axes[1].set_title('Temperature by Hour')
axes[1].set_xlabel('Hour of Day')

# Plot blood pressure
sns.boxplot(data=vitals, x=vitals.index.hour, y='blood_pressure_systolic', ax=axes[2])
axes[2].set_title('Systolic Blood Pressure by Hour')
axes[2].set_xlabel('Hour of Day')

plt.tight_layout()
plt.show()

## 3. Time Series Decomposition

Let's decompose one of our vital signs into its trend, seasonal, and residual components.

In [ ]:
# Perform seasonal decomposition on heart rate
decomposition = seasonal_decompose(
    vitals['heart_rate'],
    period=24  # 24 hours for daily seasonality
)

# Plot decomposition
fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(12, 10))
fig.suptitle('Decomposition of Heart Rate Time Series')

# Original Data
decomposition.observed.plot(ax=ax1)
ax1.set_title('Original Data')

# Trend
decomposition.trend.plot(ax=ax2)
ax2.set_title('Trend')

# Seasonal
decomposition.seasonal.plot(ax=ax3)
ax3.set_title('Seasonal')

# Residual
decomposition.resid.plot(ax=ax4)
ax4.set_title('Residual')

plt.tight_layout()
plt.show()

## 4. Handling Missing Values

Let's simulate some missing values and explore different imputation strategies.

In [ ]:
# Create copy of data and introduce missing values
vitals_missing = vitals.copy()
np.random.seed(42)

# Randomly set 10% of values to NaN
mask = np.random.random(len(vitals)) < 0.1
vitals_missing.loc[mask, 'heart_rate'] = np.nan

# Different imputation strategies
vitals_imputed = pd.DataFrame({
    'original': vitals_missing['heart_rate'],
    'forward_fill': vitals_missing['heart_rate'].fillna(method='ffill'),
    'backward_fill': vitals_missing['heart_rate'].fillna(method='bfill'),
    'linear_interpolation': vitals_missing['heart_rate'].interpolate(method='linear'),
    'time_interpolation': vitals_missing['heart_rate'].interpolate(method='time')
})

# Plot different imputation methods
fig, axes = plt.subplots(5, 1, figsize=(12, 12))
fig.suptitle('Comparison of Missing Value Imputation Methods')

for i, (method, data) in enumerate(vitals_imputed.items()):
    data.plot(ax=axes[i], title=f'{method.replace("_", " ").title()}')
    axes[i].set_xlabel('')

plt.tight_layout()
plt.show()

## 5. Correlation Analysis

Let's examine relationships between different vital signs.

In [ ]:
# Calculate correlation matrix
correlation = vitals.corr()

# Plot correlation heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation between Vital Signs')
plt.show()

# Plot relationships
sns.pairplot(vitals)
plt.show()